<a href="https://colab.research.google.com/github/CPTR295/Sample-LLMs/blob/main/LLM_architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
GPT_CONFIG_124M = {
    "vocab_size":50257, #Vocabulary Size
    "context_length":1024, #Context Length
    "emb_dim":768 , #Embedding Dimension
    "n_heads":12, #number of attention heads
    "n_layers":12, #number of layers
    "drop_rate":0.1, #Dropout rate
    "qkv_bias":False #Query-Key-Value bias
}

In [2]:
import torch
import torch.nn as nn
#Basic Architecture
class DummyGPTModel(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.tok_emb = nn.Embedding(cfg["vocab_size"],cfg['emb_dim'])
    self.pos_emb = nn.Embedding(cfg['context_length0'],cfg['emb_dim'])
    self.drop_emb = nn.Dropout(cfg['drop_rate'])
    #Placeholder for transformer blocks
    self.trf_blocks = nn.Sequential(*[DummyTransformerBlock(cfg) for _ in range(cfg['n_layers'])])
    #Placeholder for LayerNorm
    self.final_norm = DummyLayerNorm(cfg['emb_dim'])
    self.out_head = nn.Linear(cfg['emb_dim'],cfg['vocab_size'],bias=False)
  def forward(self,in_idx):
    batch_size,seq_len = in_idx.shape
    tok_embeds = self.tok_emb(in_idx)
    pos_embeds = self.pos_emb(torch.arange(seq_len,device=in_idx.device))
    x = tok_embeds + pos_embeds
    x = self.drop_emb(x)
    x = self.trf_blocks(x)
    x = self.final_norm(x)
    logits = self.out_head(x)
    return logits

class DummyTransformerBlock(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    #A simple placeholder

  def forward(self,x):
    # This block does nothing
    return x

class DummyLayerNorm(nn.Module):
  def __init__(self,normalized_shape,eps=1e-5):
    super().__init__()
    #Mimic the layernorm interface
  def forward(self,x):
    # This block does nothing
    return x

In [3]:
import tiktoken
tokenizer = tiktoken.get_encoding('gpt2')
batch =[]
txt1 = 'Every effort moves you'
txt2 = 'Every day holds a'

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch,dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [4]:
#Use above batch to try above model
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)
logits = model(batch)
print("Shape:",logits.shape)
print("Logits:",logits)

Shape: torch.Size([2, 4, 50257])
Logits: tensor([[[-1.2034,  0.3201, -0.7130,  ..., -1.5548, -0.2390, -0.4667],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6755, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0908,  0.1798, -0.9484,  ..., -1.6047,  0.2439, -0.4530],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)


In [5]:
#Normalizing
batch_sample = torch.randn(2,5)
layer = nn.Sequential(nn.Linear(5,6),nn.ReLU())
out = layer(batch_sample)
# print(out) Assume out is sample outputs
mean = out.mean(dim=-1,keepdim=True)
var = out.var(dim=-1,keepdim=True)
#print(mean)
#print(var)
out_norm = (out - mean)/torch.sqrt(var)
print(out_norm)
mean = out_norm.mean(dim=-1,keepdim=True)
var = out_norm.var(dim=-1,keepdim=True)
print(mean) #Close to 0 after normalizing
print(var) #Close to 1 after normalizing

tensor([[-0.7643,  1.7310, -0.7643,  0.4968, -0.7643,  0.0651],
        [ 1.4447,  0.9651, -0.8126, -0.8126, -0.8126,  0.0280]],
       grad_fn=<DivBackward0>)
tensor([[4.9671e-08],
        [1.9868e-08]], grad_fn=<MeanBackward1>)
tensor([[1.],
        [1.]], grad_fn=<VarBackward0>)


In [6]:
#Hence update LayerNormClass
class LayerNorm(nn.Module):
  def __init__(self,emb_dim):
    super().__init__()
    self.eps = 1e-5  #To avoid division by zero errors
    self.scale = nn.Parameter(torch.ones(emb_dim))
    self.shift = nn.Parameter(torch.zeros(emb_dim))
  def forward(self,x):
    mean = x.mean(dim=-1,keepdim=True)
    var = x.var(dim=-1,keepdim=True)
    x = (x - mean)/torch.sqrt(var + self.eps)
    x = self.scale * x + self.shift
    return x

In [7]:
out

tensor([[0.0000, 1.5463, 0.0000, 0.7815, 0.0000, 0.5140],
        [0.9592, 0.7554, 0.0000, 0.0000, 0.0000, 0.3572]],
       grad_fn=<ReluBackward0>)

In [8]:
#Trying LayerNorm
ln = LayerNorm(emb_dim=6)
out_ln = ln(out)
out_ln.mean = out_ln.mean(dim=-1, keepdim=True)
var = out_ln.var(dim=-1, unbiased=False, keepdim=True)
print(mean)
print(var)

tensor([[4.9671e-08],
        [1.9868e-08]], grad_fn=<MeanBackward1>)
tensor([[0.8333],
        [0.8333]], grad_fn=<VarBackward0>)


In [9]:
#Feed Forward with GELU activations
class GELU(nn.Module):
  def __init__(self):
    super().__init__()
  def forward(self,x):
    return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi))*(x+0.044715 * torch.pow(x,3))))

In [10]:
class FeedForward(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(cfg['emb_dim'],4*cfg['emb_dim']),
        GELU(),
        nn.Linear(4*cfg['emb_dim'],cfg['emb_dim'])
    )
  def forward(self,x):
    return self.layers(x)

In [11]:
ffn = FeedForward(GPT_CONFIG_124M)
x = torch.rand(2,3,768)
out = ffn(x)
out.shape

torch.Size([2, 3, 768])

In [12]:
#Shortcut / residual connection - mainly used to deal with vanishing gradient
class ExampleDeepNeuralNetwork(nn.Module):
  def __init__(self,layer_sizes,use_shortcut):
    super().__init__()
    self.use_shortcut = use_shortcut
    self.layers = nn.ModuleList([
        nn.Sequential(nn.Linear(layer_sizes[0],layer_sizes[1]),GELU()),
        nn.Sequential(nn.Linear(layer_sizes[1],layer_sizes[2]),GELU()),
        nn.Sequential(nn.Linear(layer_sizes[2],layer_sizes[3]),GELU()),
        nn.Sequential(nn.Linear(layer_sizes[3],layer_sizes[4]),GELU()),
        nn.Sequential(nn.Linear(layer_sizes[4],layer_sizes[5]),GELU())
    ])
  def forward(self,x):
    for layer in self.layers:
      layer_output = layer(x)
      if self.use_shortcut and x.shape == layer_output.shape:
        x = x + layer_output
      else:
        x = layer_output
    return x

In [13]:
def print_gradients(model,x):
  output = model(x)
  target = torch.tensor([[0.]])
  loss = nn.MSELoss()
  loss = loss(output,target)
  loss.backward()
  for name,param in model.named_parameters():
    if 'weight' in name:
      print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")

In [14]:
layer_sizes = [3,3,3,3,3,1]
sample_input = torch.tensor([[1.,0.,-1.]])
model_without_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes,use_shortcut=False
)
print_gradients(model_without_shortcut,sample_input)

layers.0.0.weight has gradient mean of 0.0002399790391791612
layers.1.0.weight has gradient mean of 0.00022850162349641323
layers.2.0.weight has gradient mean of 0.0006631719297729433
layers.3.0.weight has gradient mean of 0.004308415576815605
layers.4.0.weight has gradient mean of 0.07094331830739975


In [15]:
model_without_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes,use_shortcut=True
)
print_gradients(model_without_shortcut,sample_input)

layers.0.0.weight has gradient mean of 0.1509636640548706
layers.1.0.weight has gradient mean of 0.4411108195781708
layers.2.0.weight has gradient mean of 0.32774630188941956
layers.3.0.weight has gradient mean of 0.05380108579993248
layers.4.0.weight has gradient mean of 2.0143239498138428


In [19]:
#Multihead attention - refer previous notebook
class MultiHeadAttention(nn.Module):
  def __init__(self,d_in,d_out,context_length,dropout,num_heads,qkv_bias=False):
    super().__init__()
    assert d_out % num_heads == 0,"d_out must be divisible by num_heads"
    self.d_out = d_out
    self.num_heads = num_heads
    self.head_dim = d_out // num_heads
    self.W_query = nn.Linear(d_in,d_out,bias=qkv_bias)
    self.W_key = nn.Linear(d_in,d_out,bias=qkv_bias)
    self.W_value = nn.Linear(d_in,d_out,bias=qkv_bias)
    self.out_proj = nn.Linear(d_out,d_out)
    self.dropout = nn.Dropout(dropout)
    self.register_buffer('mask',torch.triu(torch.ones(context_length,context_length),diagonal=1))
  def forward(self,x):
    b,num_tokens,d_in = x.shape
    keys  = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)
    #unroll last dim (b,num_tokens,d_out) -> (b,num_tokens,num_heads,head_dim)
    keys = keys.view(b,num_tokens,self.num_heads,self.head_dim)
    queries = queries.view(b,num_tokens,self.num_heads,self.head_dim)
    values = values.view(b,num_tokens,self.num_heads,self.head_dim)

    #Transpose (b,num_tokens,num_head,head_dim) -> (n,num_head,num_token,head_dim)
    keys = keys.transpose(1,2)
    queries = queries.transpose(1,2)
    values = values.transpose(1,2)

    #Scaled dot product
    attn_scores = queries @ keys.transpose(2,3)
    mask_bool = self.mask.bool()[:num_tokens,:num_tokens]
    attn_scores.masked_fill_(mask_bool,-torch.inf)
    attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)
    attn_weights = self.dropout(attn_weights)
    context_vec = (attn_weights @ values).transpose(1,2)
    context_vec = context_vec.contiguous().view(b,num_tokens,self.d_out)
    context_vec = self.out_proj(context_vec)
    return context_vec

In [28]:
#Transformer block
class TransformerBlock(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.att = MultiHeadAttention(
        d_in = cfg['emb_dim'],
        d_out = cfg['emb_dim'],
        context_length = cfg['context_length'],
        dropout = cfg['drop_rate'],
        num_heads = cfg['n_heads'],
        qkv_bias = cfg['qkv_bias']
    )
    self.ff = FeedForward(cfg)
    self.norm1 = LayerNorm(cfg['emb_dim'])
    self.norm2 = LayerNorm(cfg['emb_dim'])
    self.drop_shortcut = nn.Dropout(cfg['drop_rate'])
  def forward(self,x):
    shortcut  =x  #Residual connection for attntion layer
    x = self.norm1(x)
    x = self.att(x)
    x = self.drop_shortcut(x)
    x = x + shortcut
    shortcut = x
    x = self.norm2(x) # Residual connection for feed forward layer
    x = self.ff(x)
    x = self.drop_shortcut(x)
    x = x + shortcut
    return x

In [29]:
x = torch.rand(2,4,768)
block = TransformerBlock(GPT_CONFIG_124M)
out = block(x)
out.shape

torch.Size([2, 4, 768])

In [30]:
#GPT Model
class GPTModel(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.tok_emb = nn.Embedding(cfg['vocab_size'],cfg['emb_dim'])
    self.pos_emb = nn.Embedding(cfg['context_length'],cfg['emb_dim'])
    self.drop_emb = nn.Dropout(cfg['drop_rate'])
    self.trf_blocks = nn.Sequential(
        *[TransformerBlock(cfg) for _ in range(cfg['n_layers'])]
    )
    self.final_norm = LayerNorm(cfg['emb_dim'])
    self.out_head = nn.Linear(cfg['emb_dim'],cfg['vocab_size'],bias=False)
  def forward(self,in_idx):
    batch_size,seq_len = in_idx.shape
    tok_embeds = self.tok_emb(in_idx)
    pos_embeds = self.pos_emb(torch.arange(seq_len,device=in_idx.device))
    x = tok_embeds + pos_embeds
    x = self.drop_emb(x)
    x = self.trf_blocks(x)
    x = self.final_norm(x)
    logits = self.out_head(x)
    return logits

In [31]:
batch

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

In [32]:
model = GPTModel(GPT_CONFIG_124M)
logits = model(batch)
print('Shape:',out.shape)
print(out)

Shape: torch.Size([2, 4, 768])
tensor([[[ 0.5419, -0.3412,  0.6684,  ...,  1.0624,  0.4397,  0.6347],
         [ 0.4844,  0.3683, -0.1617,  ...,  0.6710,  1.0371, -1.0405],
         [ 0.1037,  0.5249,  0.0059,  ...,  0.4445,  0.1364,  0.5172],
         [-0.1322,  0.6651,  0.2025,  ...,  0.8655,  0.2546,  0.0522]],

        [[ 0.5265,  1.2562,  0.4143,  ...,  0.5432,  0.4622, -0.2847],
         [ 0.5156,  1.2844,  0.2858,  ...,  0.3145,  0.7581,  0.2349],
         [ 0.4705,  1.0196,  0.1266,  ...,  0.8944,  0.6177,  1.0057],
         [ 0.9343,  0.8720, -0.0556,  ...,  0.3674,  0.2153,  0.8616]]],
       grad_fn=<AddBackward0>)


In [34]:
total_params = sum(p.numel() for p in model.parameters())
total_params

163009536

In [35]:
print("Token embedding layer shape:", model.tok_emb.weight.shape)
print("Output layer shape:", model.out_head.weight.shape)

Token embedding layer shape: torch.Size([50257, 768])
Output layer shape: torch.Size([50257, 768])


In [52]:
#Generate text
def generate_text_simple(model,idx,max_new_tokens,context_size):
  for _ in range(max_new_tokens):
    idx_cond = idx[:,-context_size:] #Crop till context size
    #print('idx_cond shape',idx_cond.shape)
    with torch.no_grad():
      logits = model(idx_cond)
      #print('Logits Shape',logits.shape)
    logits = logits[:,-1,:]#For every batch get only last token (batch,n_tokens,vocab_size) -> (batch,vocab_size)
    #print('Logit Shape',logits.shape)
    probas = torch.softmax(logits,dim=-1)
    #print("probas shape",probas.shape)
    idx_next = torch.argmax(probas,dim=-1,keepdim=True)
    #print('idx_next shape',idx_next.shape)
    #print('idx shape',idx.shape)
    idx = torch.cat((idx,idx_next),dim=1)
  return idx

In [53]:
start_context = 'Hello, I am'
encoded = tokenizer.encode(start_context)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print(encoded)
print(encoded_tensor)
print(encoded_tensor.shape)

[15496, 11, 314, 716]
tensor([[15496,    11,   314,   716]])
torch.Size([1, 4])


In [54]:
model.eval()
out = generate_text_simple(
    model = model,
    idx = encoded_tensor,
    max_new_tokens = 10,
    context_size = GPT_CONFIG_124M['context_length']
)
print(out)

tensor([[15496,    11,   314,   716, 49073, 34381, 22223, 31603, 34397, 29378,
           815, 24566,  8633,  2077]])


In [55]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
decoded_text

'Hello, I amAbyss Dice Centers Saul carrots shareholder should falsely dict taken'